In [1]:
# Cell 1 — Frozen config, load all Processed 30-s windows, and validate coverage
from __future__ import annotations

import os
import re
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed


def locate_phase1(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src' / 'dataloader' / 'loader.py').is_file() and (candidate / 'segmentated_data').is_dir():
            return candidate
        nested = candidate / 'phase1'
        if (nested / 'src' / 'dataloader' / 'loader.py').is_file() and (nested / 'segmentated_data').is_dir():
            return nested
    raise FileNotFoundError('Could not locate phase1.')


def require(condition: bool, message: str) -> None:
    if not condition:
        raise ValueError(message)


def session_number(value: str | Path) -> int:
    match = re.search(r'(\d+)$', Path(value).stem)
    return int(match.group(1)) if match else 10**9


def strict_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    values = series.astype(str).str.strip().str.casefold()
    mapping = {'true': True, 'false': False, '1': True, '0': False}
    require(values.isin(mapping).all(), f'Invalid Boolean values in {series.name}.')
    return values.map(mapping).astype(bool)


PHASE1_DIR = locate_phase1(Path.cwd())
SRC_DIR = PHASE1_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from chaos.lyapunov import compute_rosenstein_lle, estimate_mean_period
from dataloader.loader import get_data
from prediction.simplex_projection import prediction_curve
from rqa.rqa import run_rqa

DATA_DIR = PHASE1_DIR / 'segmentated_data' / '30s_dhdata'
INDEX_PATH = DATA_DIR / 'segments_index.csv'
RESULT_ROOT = PHASE1_DIR / 'results' / '30s_window'
SIMPLEX_DIR = RESULT_ROOT / 'simplex'
RQA_DIR = RESULT_ROOT / 'rqa'
LLE_DIR = RESULT_ROOT / 'lle'
for directory in (SIMPLEX_DIR, RQA_DIR, LLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DURATION_S = 30
REPRESENTATION = 'Processed'
STATE_NAMES = {0: 'Awake', 1: 'Drowsy'}
M = 8
TAU_S = 0.16
HORIZON_SECONDS = np.array([0.04, 0.08, 0.12, 0.16, 0.20, 0.28, 0.40, 0.60, 0.80, 1.00, 1.20, 1.60, 2.00, 2.40, 2.80, 3.20, 3.60, 4.00])
SIMPLEX_THEILER_S = 1.0
RQA_TARGET_RR = 0.02
RQA_L_MIN = 2
RQA_V_MIN = 2
RQA_RR_TOLERANCE = 5e-5
LLE_FIT_START_S = 0.80
LLE_FIT_END_S = 1.30
LLE_MAX_FOLLOW_S = 5.0
LLE_MIN_INITIAL_PAIRS = 50
LLE_MIN_FIT_PAIRS = 30
LLE_MIN_R2 = 0.90
N_JOBS = min(6, os.cpu_count() or 1)

session_paths = sorted(DATA_DIR.glob('sample_*.npz'), key=session_number)
require(session_paths, f'No session archives found in {DATA_DIR}.')
index = pd.read_csv(INDEX_PATH)
required_index_columns = {'npz_file', 'window_id', 'window_size_s', 'label', 'fs', 'n_samples', 'stationarity_pass_processed', 'window_30_id', 'parent_window_60_id', 'subwindow'}
require(required_index_columns.issubset(index.columns), f'Index missing columns: {sorted(required_index_columns - set(index.columns))}')
index['stationarity_pass_processed'] = strict_bool(index['stationarity_pass_processed'])
index['session_key'] = index['npz_file'].map(lambda value: Path(value).stem)
index['state'] = index['label'].map(STATE_NAMES)
eligible_index = index.loc[index['window_size_s'].eq(DURATION_S) & index['stationarity_pass_processed']].copy()
require(eligible_index['state'].notna().all(), 'Unsupported state label in the 30-s index.')
identity = ['session_key', 'state', 'window_id']
require(not eligible_index.duplicated(identity).any(), 'Duplicate eligible input identities.')
index_lookup = eligible_index.set_index(identity)
require(index_lookup.index.is_unique, 'Eligible input index is not unique.')

tasks = []
for session_path in session_paths:
    session = session_path.stem
    awake, drowsy = get_data(session, data_dir=DATA_DIR, window_sizes=DURATION_S, stationarity='processed')
    for state, batch in (('Awake', awake[DURATION_S]), ('Drowsy', drowsy[DURATION_S])):
        for array_index, window_id in enumerate(batch['window_id']):
            key = (session, state, int(window_id))
            require(key in index_lookup.index, f'Loaded window absent from index: {key}')
            row = index_lookup.loc[key]
            signal = np.asarray(batch['processed'][array_index], dtype=float)
            fs = float(batch['fs'])
            require(signal.ndim == 1 and signal.size > 0, f'Invalid signal shape: {key}')
            require(np.isfinite(signal).all(), f'Non-finite Processed signal: {key}')
            require(signal.size == int(row['n_samples']), f'Sample-count mismatch: {key}')
            require(np.isclose(fs, float(row['fs'])), f'Sampling-rate mismatch: {key}')
            tasks.append({
                'session': session, 'session_id': session_number(session), 'state': state,
                'window_id': int(window_id), 'window_30_id': str(row['window_30_id']),
                'parent_window_60_id': str(row['parent_window_60_id']), 'subwindow': str(row['subwindow']),
                'fs': fs, 'n_samples': signal.size, 'signal': signal,
            })

task_index = pd.DataFrame([{key: value for key, value in task.items() if key != 'signal'} for task in tasks])
loaded_keys = set(task_index[['session', 'state', 'window_id']].itertuples(index=False, name=None))
expected_keys = set(eligible_index[['session_key', 'state', 'window_id']].itertuples(index=False, name=None))
require(loaded_keys == expected_keys, f'Loaded/index key mismatch: loaded-only={len(loaded_keys - expected_keys)}, index-only={len(expected_keys - loaded_keys)}')
require(not task_index.duplicated(['session', 'state', 'window_id']).any(), 'Duplicate loaded window identities.')
require(task_index['session'].nunique() == len(session_paths), 'Session coverage mismatch.')

dataset_summary = pd.DataFrame({
    'quantity': ['sessions', 'total_windows', 'Awake_windows', 'Drowsy_windows', 'workers'],
    'value': [task_index['session'].nunique(), len(task_index), task_index['state'].eq('Awake').sum(), task_index['state'].eq('Drowsy').sum(), N_JOBS],
})
config_summary = pd.DataFrame([
    {'method': 'Simplex', 'frozen_config': 'm=8; tau=0.16 s; 18 horizons=0.04-4.00 s; Theiler=1.0 s; k=m+1; Euclidean; exponential weighting'},
    {'method': 'RQA', 'frozen_config': 'm=8; tau=0.16 s; Euclidean; target RR=0.02; Theiler=(m-1)*tau; lmin=vmin=2'},
    {'method': 'LLE', 'frozen_config': 'm=8; tau=0.16 s; spectral-mean-period Theiler; fit=0.80-1.30 s; max follow=5 s; R2>=0.90'},
])
sampling_summary = task_index.assign(sampling_rate_group_hz=task_index['fs'].round().astype(int)).groupby(['state', 'sampling_rate_group_hz']).size().rename('n_windows').reset_index()
display(dataset_summary, sampling_summary, config_summary)
print(f'Input validation: PASS | source={DATA_DIR} | output={RESULT_ROOT}')

,quantity,value
0,sessions,20
1,total_windows,1802
2,Awake_windows,1192
3,Drowsy_windows,610
4,workers,6


,state,sampling_rate_group_hz,n_windows
0,Awake,25,1144
1,Awake,50,48
2,Drowsy,25,596
3,Drowsy,50,14


,method,frozen_config
0,Simplex,m=8; tau=0.16 s; 18 horizons=0.04-4.00 s; Thei...
1,RQA,m=8; tau=0.16 s; Euclidean; target RR=0.02; Th...
2,LLE,m=8; tau=0.16 s; spectral-mean-period Theiler;...


Input validation: PASS | source=/home/vutu0809/Desktop/NTSA_Foundation/phase1/segmentated_data/30s_dhdata | output=/home/vutu0809/Desktop/NTSA_Foundation/phase1/results/30s_window


In [2]:
# Cell 2 — Run the frozen Simplex, RQA, and Rosenstein LLE cores on every window
def unit_scale(_values: np.ndarray) -> float:
    return 1.0


def evaluate_window(task: dict) -> dict[str, dict]:
    signal = np.asarray(task['signal'], dtype=float)
    fs = float(task['fs'])
    tau_samples = max(int(round(TAU_S * fs)), 1)
    base = {
        'session': task['session'], 'session_id': task['session_id'], 'state': task['state'],
        'window_id': task['window_id'], 'window_30_id': task['window_30_id'],
        'parent_window_60_id': task['parent_window_60_id'], 'subwindow': task['subwindow'],
        'duration_s': DURATION_S, 'n_samples': task['n_samples'],
    }

    simplex_row = {
        **base, 'window_size_s': DURATION_S, 'representation': REPRESENTATION, 'fs': fs,
        'm': M, 'tau_seconds': TAU_S, 'tau_samples': tau_samples,
        'theiler_seconds': SIMPLEX_THEILER_S, 'theiler_samples': int(round(SIMPLEX_THEILER_S * fs)),
        'n_horizons': len(HORIZON_SECONDS), 'horizon_min_s': float(HORIZON_SECONDS.min()),
        'horizon_max_s': float(HORIZON_SECONDS.max()), 'Mean_CC': np.nan, 'Mean_NRMSE': np.nan,
        'mean_valid_fraction': 0.0, 'minimum_valid_fraction': 0.0,
        'valid': False, 'status': 'failed', 'failure_reason': '',
    }
    try:
        scale = float(np.std(signal))
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError('Signal must have non-zero finite variance.')
        standardized = (signal - np.mean(signal)) / scale
        horizons = np.rint(HORIZON_SECONDS * fs).astype(int)
        curve = prediction_curve(
            signal=standardized, tau=tau_samples, m=M, horizons=horizons,
            theiler_window=simplex_row['theiler_samples'], scale_function=unit_scale,
        )
        cc = np.asarray([metric.cc for metric in curve], dtype=float)
        nrmse = np.asarray([metric.nrmse for metric in curve], dtype=float)
        n_valid = np.asarray([metric.n_valid for metric in curve], dtype=int)
        n_targets = signal.size - (M - 1) * tau_samples - horizons
        valid_fraction = n_valid / n_targets
        finite_metrics = bool(np.isfinite(cc).all() and np.isfinite(nrmse).all())
        simplex_row.update({
            'Mean_CC': float(np.mean(cc)) if finite_metrics else np.nan,
            'Mean_NRMSE': float(np.mean(nrmse)) if finite_metrics else np.nan,
            'mean_valid_fraction': float(np.mean(valid_fraction)),
            'minimum_valid_fraction': float(np.min(valid_fraction)),
            'valid': finite_metrics, 'status': 'ok' if finite_metrics else 'invalid_metric',
            'failure_reason': '' if finite_metrics else 'non_finite_horizon_metric',
        })
    except Exception as exc:
        simplex_row['failure_reason'] = f'{type(exc).__name__}: {exc}'

    rqa_row = {
        **base, 'representation': REPRESENTATION, 'window_size': DURATION_S, 'fs': fs,
        'm': M, 'tau_seconds': TAU_S, 'tau_samples': tau_samples,
        'theiler_samples': (M - 1) * tau_samples, 'distance_metric': 'euclidean',
        'l_min': RQA_L_MIN, 'v_min': RQA_V_MIN, 'epsilon': np.nan,
        'target_rr': RQA_TARGET_RR, 'achieved_rr': np.nan, 'zero_distance_fraction': np.nan,
        'rr_exact': False, 'DET': np.nan, 'Lmean': np.nan, 'ENTR': np.nan, 'Lmax': np.nan,
        'LAM': np.nan, 'TT': np.nan, 'Vmax': np.nan, 'n_diagonal_lines': 0,
        'n_vertical_lines': 0, 'valid': False, 'status': 'failed', 'failure_reason': '',
    }
    try:
        result = run_rqa(signal, m=M, tau=tau_samples, l_min=RQA_L_MIN, v_min=RQA_V_MIN, target_rr=RQA_TARGET_RR)
        rqa_row.update({
            'theiler_samples': result.theiler, 'epsilon': result.epsilon,
            'achieved_rr': result.achieved_rr, 'zero_distance_fraction': result.zero_distance_fraction,
            'rr_exact': result.rr_exact, 'DET': result.det, 'Lmean': result.l_mean,
            'ENTR': result.entr, 'Lmax': result.l_max, 'LAM': result.lam, 'TT': result.tt,
            'Vmax': result.v_max, 'n_diagonal_lines': result.n_diagonal_lines,
            'n_vertical_lines': result.n_vertical_lines,
        })
        numeric = np.asarray([result.epsilon, result.achieved_rr, result.det, result.l_mean, result.entr, result.l_max, result.lam, result.tt, result.v_max], dtype=float)
        checks = {
            'finite': np.isfinite(numeric).all(), 'epsilon_positive': result.epsilon > 0,
            'rr_tolerance': abs(result.achieved_rr - RQA_TARGET_RR) <= RQA_RR_TOLERANCE,
            'det_lam_bounds': 0 <= result.det <= 1 and 0 <= result.lam <= 1,
            'nonnegative_metrics': min(result.l_mean, result.entr, result.l_max, result.tt, result.v_max) >= 0,
            'line_support': result.n_diagonal_lines > 0 and result.n_vertical_lines > 0,
            'line_maxima': result.l_max >= result.l_mean and result.v_max >= result.tt,
        }
        failed_checks = [name for name, passed in checks.items() if not passed]
        rqa_row.update({
            'valid': not failed_checks, 'status': 'ok' if not failed_checks else 'qc_failed',
            'failure_reason': '' if not failed_checks else ';'.join(failed_checks),
        })
    except Exception as exc:
        rqa_row['failure_reason'] = f'{type(exc).__name__}: {exc}'

    lle_row = {
        **base, 'session': task['session_id'], 'window_size_s': DURATION_S, 'representation': REPRESENTATION.lower(),
        'sampling_rate_hz': fs, 'm': M, 'tau_s': TAU_S, 'tau_samples': tau_samples,
        'theiler_s': np.nan, 'theiler_samples': np.nan, 'fit_start_s': LLE_FIT_START_S,
        'fit_end_s': LLE_FIT_END_S, 'fit_duration_s': LLE_FIT_END_S - LLE_FIT_START_S,
        'max_follow_s': LLE_MAX_FOLLOW_S, 'lle_1_per_s': np.nan, 'fit_r2': np.nan,
        'n_embedded': max(signal.size - (M - 1) * tau_samples, 0), 'n_pairs_initial': 0,
        'n_pairs_fit_min': 0, 'analysis_included': True, 'signal_finite': bool(np.isfinite(signal).all()),
        'valid': False, 'qc_reason': '',
    }
    try:
        theiler_s = float(estimate_mean_period(signal, fs))
        result = compute_rosenstein_lle(
            signal, sampling_rate=fs, m=M, tau_samples=tau_samples,
            fit_start_s=LLE_FIT_START_S, fit_end_s=LLE_FIT_END_S,
            max_follow_s=LLE_MAX_FOLLOW_S, theiler_s=theiler_s,
            min_initial_pairs=LLE_MIN_INITIAL_PAIRS, min_fit_pairs=LLE_MIN_FIT_PAIRS, min_r2=LLE_MIN_R2,
        )
        lle_row.update({
            'theiler_s': theiler_s, 'theiler_samples': result.theiler_samples,
            'lle_1_per_s': result.lle, 'fit_r2': result.fit_r2, 'n_embedded': result.n_embedded,
            'n_pairs_initial': result.n_pairs_initial, 'n_pairs_fit_min': result.n_pairs_fit_min,
            'valid': result.valid, 'qc_reason': result.qc_reason,
        })
    except Exception as exc:
        lle_row['qc_reason'] = f'exception_{type(exc).__name__}: {exc}'

    return {'simplex': simplex_row, 'rqa': rqa_row, 'lle': lle_row}


run_started = perf_counter()
evaluations = Parallel(n_jobs=N_JOBS, backend='loky', batch_size=1, pre_dispatch=N_JOBS, verbose=5)(
    delayed(evaluate_window)(task) for task in tasks
)
runtime_s = perf_counter() - run_started
simplex_results = pd.DataFrame([item['simplex'] for item in evaluations])
rqa_results = pd.DataFrame([item['rqa'] for item in evaluations])
lle_results = pd.DataFrame([item['lle'] for item in evaluations])
print(f'Completed all three pipelines for {len(tasks):,} windows in {runtime_s / 60:.2f} min using {N_JOBS} workers.')

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.


[Parallel(n_jobs=6)]: Done  12 tasks      | elapsed:    4.0s


[Parallel(n_jobs=6)]: Done  66 tasks      | elapsed:   16.7s


[Parallel(n_jobs=6)]: Done 156 tasks      | elapsed:   27.8s


[Parallel(n_jobs=6)]: Done 282 tasks      | elapsed:   44.0s


...

[Parallel(n_jobs=6)]: Done 1146 tasks      | elapsed:  2.6min


[Parallel(n_jobs=6)]: Done 1452 tasks      | elapsed:  3.3min


[Parallel(n_jobs=6)]: Done 1794 tasks      | elapsed:  4.1min


Completed all three pipelines for 1,802 windows in 4.08 min using 6 workers.


[Parallel(n_jobs=6)]: Done 1802 out of 1802 | elapsed:  4.1min finished


In [3]:
# Cell 3 — Validate one-row-per-window outputs, save compatible CSVs, and report QC
KEY_COLUMNS = ['session_id', 'state', 'window_id', 'duration_s']
LINEAGE_COLUMNS = ['window_30_id', 'parent_window_60_id', 'subwindow']
expected_result_keys = set(task_index.assign(duration_s=DURATION_S)[KEY_COLUMNS].itertuples(index=False, name=None))


def validate_identity(frame: pd.DataFrame, method: str) -> None:
    require(len(frame) == len(tasks), f'{method}: expected {len(tasks)} rows, found {len(frame)}.')
    require(not frame.duplicated(KEY_COLUMNS).any(), f'{method}: duplicate window identities.')
    actual = set(frame[KEY_COLUMNS].itertuples(index=False, name=None))
    require(actual == expected_result_keys, f'{method}: input/result key mismatch.')
    require(frame['duration_s'].eq(DURATION_S).all(), f'{method}: duration_s is not fixed at 30.')
    require(frame['state'].isin(STATE_NAMES.values()).all(), f'{method}: invalid state label.')
    require(frame[LINEAGE_COLUMNS].notna().all().all(), f'{method}: lineage metadata is incomplete.')


for method, frame in (('Simplex', simplex_results), ('RQA', rqa_results), ('LLE', lle_results)):
    validate_identity(frame, method)

require(simplex_results['m'].eq(M).all() and np.allclose(simplex_results['tau_seconds'], TAU_S), 'Simplex embedding config changed.')
require(np.allclose(simplex_results['theiler_seconds'], SIMPLEX_THEILER_S), 'Simplex Theiler changed.')
require(simplex_results['n_horizons'].eq(len(HORIZON_SECONDS)).all(), 'Simplex horizon grid is incomplete.')
require(simplex_results.loc[simplex_results['valid'], ['Mean_CC', 'Mean_NRMSE']].notna().all().all(), 'A valid Simplex row has missing metrics.')
require(rqa_results['m'].eq(M).all() and np.allclose(rqa_results['tau_seconds'], TAU_S), 'RQA embedding config changed.')
require(np.array_equal(rqa_results['theiler_samples'].to_numpy(int), (M - 1) * rqa_results['tau_samples'].to_numpy(int)), 'RQA Theiler rule changed.')
require(np.allclose(rqa_results['target_rr'], RQA_TARGET_RR), 'RQA target RR changed.')
require(rqa_results['l_min'].eq(RQA_L_MIN).all() and rqa_results['v_min'].eq(RQA_V_MIN).all(), 'RQA line thresholds changed.')
require(rqa_results.loc[rqa_results['valid'], ['DET', 'Lmean', 'ENTR', 'Lmax', 'LAM', 'TT', 'Vmax']].notna().all().all(), 'A valid RQA row has missing metrics.')
require(lle_results['m'].eq(M).all() and np.allclose(lle_results['tau_s'], TAU_S), 'LLE embedding config changed.')
require(np.allclose(lle_results['fit_start_s'], LLE_FIT_START_S) and np.allclose(lle_results['fit_end_s'], LLE_FIT_END_S), 'LLE fit interval changed.')
lle_finite = np.isfinite(lle_results[['lle_1_per_s', 'fit_r2']].to_numpy(dtype=float)).all(axis=1)
lle_expected_valid = lle_results['analysis_included'].astype(bool) & lle_results['signal_finite'].astype(bool) & lle_finite & lle_results['fit_r2'].ge(LLE_MIN_R2) & lle_results['qc_reason'].eq('ok')
require(lle_results['valid'].astype(bool).eq(lle_expected_valid).all(), 'LLE valid flags differ from frozen primary QC.')

simplex_columns = ['session', 'session_id', 'state', 'window_id', 'window_30_id', 'parent_window_60_id', 'subwindow', 'duration_s', 'window_size_s', 'representation', 'fs', 'n_samples', 'm', 'tau_seconds', 'tau_samples', 'theiler_seconds', 'theiler_samples', 'n_horizons', 'horizon_min_s', 'horizon_max_s', 'Mean_CC', 'Mean_NRMSE', 'mean_valid_fraction', 'minimum_valid_fraction', 'valid', 'status', 'failure_reason']
rqa_columns = ['session', 'session_id', 'representation', 'state', 'window_size', 'duration_s', 'window_id', 'window_30_id', 'parent_window_60_id', 'subwindow', 'fs', 'n_samples', 'm', 'tau_seconds', 'tau_samples', 'theiler_samples', 'distance_metric', 'l_min', 'v_min', 'epsilon', 'target_rr', 'achieved_rr', 'zero_distance_fraction', 'rr_exact', 'DET', 'Lmean', 'ENTR', 'Lmax', 'LAM', 'TT', 'Vmax', 'n_diagonal_lines', 'n_vertical_lines', 'valid', 'status', 'failure_reason']
lle_columns = ['session', 'session_id', 'window_id', 'window_30_id', 'parent_window_60_id', 'subwindow', 'state', 'duration_s', 'window_size_s', 'representation', 'sampling_rate_hz', 'n_samples', 'm', 'tau_s', 'tau_samples', 'theiler_s', 'theiler_samples', 'fit_start_s', 'fit_end_s', 'fit_duration_s', 'max_follow_s', 'lle_1_per_s', 'fit_r2', 'n_embedded', 'n_pairs_initial', 'n_pairs_fit_min', 'analysis_included', 'signal_finite', 'valid', 'qc_reason']
simplex_results = simplex_results[simplex_columns].sort_values(KEY_COLUMNS).reset_index(drop=True)
rqa_results = rqa_results[rqa_columns].sort_values(KEY_COLUMNS).reset_index(drop=True)
lle_results = lle_results[lle_columns].sort_values(KEY_COLUMNS).reset_index(drop=True)


def write_csv_atomic(frame: pd.DataFrame, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


SIMPLEX_PATH = SIMPLEX_DIR / 'processed_simplex_window_results_30s.csv'
RQA_PATH = RQA_DIR / 'rqa_window_level_30s.csv'
LLE_PATH = LLE_DIR / 'all_sessions_processed_rosenstein_lle_30s.csv'
for frame, path in ((simplex_results, SIMPLEX_PATH), (rqa_results, RQA_PATH), (lle_results, LLE_PATH)):
    write_csv_atomic(frame, path)
    reloaded = pd.read_csv(path)
    require(len(reloaded) == len(tasks), f'Reload row-count mismatch: {path.name}')
    require(not reloaded.duplicated(KEY_COLUMNS).any(), f'Reload duplicate identities: {path.name}')

simplex_valid = int(simplex_results['valid'].sum())
rqa_valid = int(rqa_results['valid'].sum())
lle_valid = int(lle_results['valid'].sum())
lle_failed = len(lle_results) - lle_valid
method_summary = pd.DataFrame([
    {'method': 'Simplex', 'total_windows': len(simplex_results), 'valid_windows': simplex_valid, 'invalid_windows': len(simplex_results) - simplex_valid, 'missing_primary_metrics': int(simplex_results[['Mean_CC', 'Mean_NRMSE']].isna().any(axis=1).sum())},
    {'method': 'RQA', 'total_windows': len(rqa_results), 'valid_windows': rqa_valid, 'invalid_windows': len(rqa_results) - rqa_valid, 'missing_primary_metrics': int(rqa_results['DET'].isna().sum())},
    {'method': 'LLE', 'total_windows': len(lle_results), 'valid_windows': lle_valid, 'invalid_windows': lle_failed, 'missing_primary_metrics': int(lle_results['lle_1_per_s'].isna().sum())},
])
lle_qc_reasons = lle_results.groupby(['valid', 'qc_reason'], dropna=False).size().rename('n_windows').reset_index()
display(method_summary, lle_qc_reasons)
print(
    f'Total 30-s windows: {len(tasks):,} | Sessions: {task_index["session"].nunique()} | '
    f'Awake: {task_index["state"].eq("Awake").sum():,} | Drowsy: {task_index["state"].eq("Drowsy").sum():,}\n'
    f'Simplex valid: {simplex_valid:,} | RQA valid: {rqa_valid:,} | LLE valid: {lle_valid:,}\n'
    f'LLE QC failures: {lle_failed:,}/{len(lle_results):,} ({100 * lle_failed / len(lle_results):.1f}%)\n'
    f'Saved:\n- {SIMPLEX_PATH}\n- {RQA_PATH}\n- {LLE_PATH}\nValidation: PASS'
)

,method,total_windows,valid_windows,invalid_windows,missing_primary_metrics
0,Simplex,1802,1802,0,0
1,RQA,1802,1802,0,0
2,LLE,1802,1638,164,0


,valid,qc_reason,n_windows
0,False,low_fit_r2,164
1,True,ok,1638


Total 30-s windows: 1,802 | Sessions: 20 | Awake: 1,192 | Drowsy: 610
Simplex valid: 1,802 | RQA valid: 1,802 | LLE valid: 1,638
LLE QC failures: 164/1,802 (9.1%)
Saved:
- /home/vutu0809/Desktop/NTSA_Foundation/phase1/results/30s_window/simplex/processed_simplex_window_results_30s.csv
- /home/vutu0809/Desktop/NTSA_Foundation/phase1/results/30s_window/rqa/rqa_window_level_30s.csv
- /home/vutu0809/Desktop/NTSA_Foundation/phase1/results/30s_window/lle/all_sessions_processed_rosenstein_lle_30s.csv
Validation: PASS
